# Creating Videos
This notebook covers how to use the headset_localization package to create video visualizations of the localizers

In [ ]:
%load_ext autoreload
%autoreload 2
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)

from headset_localization import *
from shared import CompleteRobotScan

In [ ]:
robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"

robot_scan = CompleteRobotScan.from_folder(robot_data_folder_location)
robot_env = Scanned3dEnvironment.from_gathered_robot_data(
        robot_data = robot_scan,
        number_of_sampled_datapoints=10,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=True)
)

labeled_headset_data = bind_headset_recording_to_scan(
        headset_data = HeadsetRecording.from_vrs_file(vrs_file_location),
        robot_data = robot_scan
)

visualize_loaded_data = False

if visualize_loaded_data:
    visualize_robot_camera_environment_combo(robot_env=robot_env, headset_data=labeled_headset_data)

### PnP localizer video generation

In [ ]:
pnp_video_predictor = PnPLocalizer(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(display_matching=True, crop_augmentations=[0.4]),
)

init_predictor_grade = PredictionOnDataset(
    predictor = pnp_video_predictor,
    headset_data = labeled_headset_data,
    vid_gen=VideoGenerator(fps=20, style_config=FeatureStyleConfig(connection_line_alpha=0.4)),
    video_save_location="test_pnp.mp4"
)
init_predictor_grade.print_summary()

### PnP+L localizer video generation

In [ ]:
pnpl_video_predictor = PnPLLocalizer(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        cam1_line_generator = LineGenerator(visualize_cleanup=False),
        cam2_line_generator=LineGenerator(
            line_cleanup_config= MultiPassLineMergingConfig(passes=[
            LineMerging2dConfig(max_angle_diff = 2, max_midpoint_dist = 3/850, max_endpoint_dist = 0.01, min_line_length = 10/850),
            LineMerging2dConfig(max_angle_diff = 3, max_midpoint_dist = 4/850, max_endpoint_dist = 0.02, min_line_length = 10/850)
            ]),
            visualize_cleanup=False
        ),
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(crop_augmentations=[0.4], extract_and_match=ExtractAndMatchLoMa()),
        line_fitting_3d_config = LineFitting3dConfig(fitting_algorithm='pca', min_number_points=4),
        line_matching_config = LineMatchingConfig(max_point_line_dist_px=10),
        debug_visualize_matching = False,
        debug_visualize_3d = False
)

init_predictor_grade = PredictionOnDataset(
    predictor = pnpl_video_predictor,
    headset_data = labeled_headset_data,
    vid_gen=VideoGenerator(fps=20, style_config=FeatureStyleConfig(
        connection_line_alpha=0.0, point_size=0, line_widht=4, point_alpha = 0.0
    )),
    video_save_location="test_pnpl.mp4"
)
init_predictor_grade.print_summary()

### Ellipsoid localizer video generation

In [ ]:
ellipsoid_video_predictor = EllipsoidLocalizer(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        matching_config=GaussianMatchingConfig(dummy_value=0.001),
        cam1_segmenter = SAM3Segmenter(Sam3Prompt(mask_threshold=0.2)),
        cam2_segmenter = YOLOv26Segmenter(),
        ellipsoid_fitter = MVEEEllipsoidFitter(contamination=0.05),
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(extract_and_match=ExtractAndLightGlue(), crop_augmentations=[0.4]),
        ellipsoid_matching_config=PointCloudMatchingConfig(min_cluster_size=2),
)

ellipsoid_predictor_grade = PredictionOnDataset(
    predictor = ellipsoid_video_predictor,
    headset_data = labeled_headset_data,
    vid_gen=VideoGenerator(fps=20, style_config=FeatureStyleConfig(unmatched_alpha=0.0, connection_line_alpha = 0.1), use_second_3d_axis=True),
    video_save_location="test_ellipsoid.mp4"
)
ellipsoid_predictor_grade.print_summary()